# Classification Tulipes / Lys

Pipeline mono-notebook : parsing -> prétraitement -> entraînement -> visualisation.

Un seul fichier Parquet est écrit, juste avant la visualisation.

> **Règles** : DataFrame uniquement · pas de `collect()` / `toPandas()` / `toList()` · Python pur = affichage seulement


## 0 · Session Spark & imports

In [3]:
import sys
import pyspark

print("Python :", sys.executable)
print("Version Python :", sys.version)
print("PySpark :", pyspark.__version__)
print("PySpark installé dans :", pyspark.__file__)

Python : c:\Users\aliam\AppData\Local\Programs\Python\Python311\python.exe
Version Python : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PySpark : 4.1.1
PySpark installé dans : c:\Users\aliam\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\__init__.py


In [4]:
import os
print(os.environ.get("JAVA_HOME"))

C:\Program Files\Java\jdk-11


In [ ]:
import sys
import os

# Java AVANT PySpark
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-11"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

# Hadoop
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

# Python utilisé par Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import io
import struct

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, FloatType, ArrayType, StringType
)

import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder


spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("TulipsLilies")
    .config("spark.sql.files.ignoreCorruptFiles", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.python.worker.faulthandler.enabled", "true")
    # Limite le batch envoyé à l'UDF à 5 images à la fois
    .config("spark.sql.execution.python.udf.arrow.enabled", "false")
    .config("spark.default.parallelism", "2")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")

KeyboardInterrupt: 

## 1 - Chemins & constantes

In [ ]:
TRAIN_PATH        = "./data/Train/"
TEST_PATH         = "./data/Test/"
OUTPUT_PREDS      = "./output/predictions/"
PARSED_TRAIN_PATH = "./output/parsed/train/"
PARSED_TEST_PATH  = "./output/parsed/test/"
TARGET_SIZE       = (32, 32)

In [ ]:
print(os.environ.get("HADOOP_HOME"))
spark.read.format("binaryFile").load(TRAIN_PATH).limit(1).show()

C:\hadoop
+----+----------------+------+-------+
|path|modificationTime|length|content|
+----+----------------+------+-------+
+----+----------------+------+-------+



## 2 - Parsing

In [ ]:
TARGET_W, TARGET_H = TARGET_SIZE

def decode_image_bytes(raw_bytes: bytes):
    try:
        from PIL import Image
        img = Image.open(io.BytesIO(raw_bytes)).convert("RGB")
        img = img.resize((TARGET_W, TARGET_H), Image.LANCZOS)
        raw = img.tobytes()
        n = len(raw)
        pixels = list(struct.unpack(f"{n}B", raw))
        return (TARGET_W, TARGET_H, 3, [float(p) for p in pixels])
    except Exception:
        return None

_decode_schema = StructType([
    StructField("width",    IntegerType(), False),
    StructField("height",   IntegerType(), False),
    StructField("channels", IntegerType(), False),
    StructField("pixels",   ArrayType(FloatType()), False),
])

decode_udf = F.udf(decode_image_bytes, _decode_schema)

def parse_images(path):
    raw = (
        spark.read.format("binaryFile")
        .option("recursiveFileLookup", "true")
        .option("pathGlobFilter", "*.{jpg,jpeg,png,JPG,PNG}")
        .load(path)
    )
    return (
        raw
        .coalesce(1)
        .select(
            F.regexp_extract(F.col("path"), r"([^/]+)$", 1).alias("image_id"),
            F.regexp_extract(F.col("path"), r"/([^/]+)/[^/]+$", 1).alias("label"),
            F.col("content").alias("raw_bytes"),
        )
        .withColumn("decoded", decode_udf(F.col("raw_bytes")))
        .filter(F.col("decoded").isNotNull())
        .select(
            "image_id", "label",
            F.col("decoded.pixels").alias("pixels"),
        )
    )

train_parsed_df = parse_images(TRAIN_PATH)
test_parsed_df  = parse_images(TEST_PATH)

print(f"Images train : {train_parsed_df.count()}")
print(f"Images test  : {test_parsed_df.count()}")
test_parsed_df.show()

c:\Users\aliam\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\udf.py:134: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


Images train : 624
Images test  : 408
+----------+-------+--------------------+
|  image_id|  label|              pixels|
+----------+-------+--------------------+
|000165.jpg|tulipes|[171.0, 136.0, 16...|
|000108.jpg|tulipes|[138.0, 155.0, 65...|
|000162.jpg|tulipes|[196.0, 108.0, 11...|
|000036.jpg|    lys|[50.0, 64.0, 32.0...|
|000052.jpg|    lys|[46.0, 57.0, 73.0...|
|000094.png|tulipes|[227.0, 233.0, 23...|
|000048.png|    lys|[1.0, 2.0, 25.0, ...|
|000080.jpg|tulipes|[110.0, 29.0, 81....|
|000079.jpg|    lys|[53.0, 51.0, 45.0...|
|000163.jpg|tulipes|[214.0, 115.0, 17...|
|000121.jpg|tulipes|[248.0, 237.0, 20...|
|000077.png|tulipes|[209.0, 199.0, 18...|
|000152.jpg|tulipes|[226.0, 152.0, 13...|
|000056.png|    lys|[236.0, 240.0, 23...|
|000084.png|tulipes|[75.0, 94.0, 25.0...|
|000096.png|    lys|[235.0, 239.0, 23...|
|000140.jpg|    lys|[80.0, 197.0, 4.0...|
|000006.jpg|tulipes|[47.0, 51.0, 41.0...|
|000178.jpg|    lys|[22.0, 35.0, 22.0...|
|000063.jpg|tulipes|[209.0, 106.0, 11.

## 3 - Prétraitement 

On garde `pixels` (RGB brut, 0-255) intact pour l'affichage futur dans Streamlit,
et on ajoute une colonne `pixels_gray` : niveaux de gris normalisés en [0.0, 1.0].

Conversion RGB -> nuances de gris : formule de luminance pondérée (standard) :
```
gray = 0.299*R + 0.587*G + 0.114*B
```

`pixels` est une liste aplatie `[R,G,B, R,G,B, ...]` de taille 64*64*3 = 12288.
L'UDF regroupe les valeurs par 3, applique la formule et renvoie les pixels en niveaux de gris

In [ ]:

def rgb_to_grayscale(pixels):
    if pixels is None:
        return None
    return [0.299 * pixels[i] + 0.587 * pixels[i+1] + 0.114 * pixels[i+2]
            for i in range(0, len(pixels), 3)]

def rgb_to_normalized_gray(pixels):
    if pixels is None:
        return None
    return [(0.299 * pixels[i] + 0.587 * pixels[i+1] + 0.114 * pixels[i+2]) / 255.0
            for i in range(0, len(pixels), 3)]

gray_raw_udf  = F.udf(rgb_to_grayscale,      ArrayType(FloatType()))
gray_norm_udf = F.udf(rgb_to_normalized_gray, ArrayType(FloatType()))

# Dataset grayscale non normalisé (0-255)
train_preprocessed_gray_df = train_base_df.withColumn("pixels_grayscale", gray_raw_udf(F.col("pixels")))
test_preprocessed_gray_df  = test_base_df.withColumn("pixels_grayscale",  gray_raw_udf(F.col("pixels")))

# Dataset grayscale normalisé (0.0-1.0)
train_preprocessed_norm_gray_df = train_base_df.withColumn("pixels_gray", gray_norm_udf(F.col("pixels")))
test_preprocessed_norm_gray_df  = test_base_df.withColumn("pixels_gray",  gray_norm_udf(F.col("pixels")))

# Vérifications
expected_len = TARGET_W * TARGET_H
print(f"Taille pixels_grayscale (attendu {expected_len}) : "
      f"{train_preprocessed_gray_df.select(F.size('pixels_grayscale')).first()[0]}")
print(f"Taille pixels_gray (attendu {expected_len}) : "
      f"{train_preprocessed_norm_gray_df.select(F.size('pixels_gray')).first()[0]}")

print("\nAperçu grayscale non normalisé :")
train_preprocessed_gray_df.select("image_id", "label", "pixels_grayscale").show(3, truncate=40)

print("Aperçu grayscale normalisé :")
train_preprocessed_norm_gray_df.select("image_id", "label", "pixels_gray").show(3, truncate=40)

NameError: name 'train_base_df' is not defined

## ML couleurs - prétraitement en bytes

## ML couleurs normalisées


## ML grayscale

In [ ]:
## 4.1 - ML : Grayscale non normalisé (0-255)

LABEL_COL   = "label"
FEATURE_COL = "pixels_grayscale"

def to_numpy(df, feature_col, label_col=LABEL_COL):
    rows = df.select("image_id", label_col, feature_col).collect()
    image_ids = [r["image_id"]    for r in rows]
    X         = np.array([r[feature_col] for r in rows], dtype=np.float32)
    y         = [r[label_col]     for r in rows]
    return image_ids, X, y

train_ids, X_train, y_train_raw = to_numpy(train_preprocessed_gray_df, FEATURE_COL)
test_ids,  X_test,  y_test_raw  = to_numpy(test_preprocessed_gray_df,  FEATURE_COL)

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test  = label_encoder.transform(y_test_raw)
print(f"Classes : {dict(enumerate(label_encoder.classes_))}")

rf_gray_raw = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_gray_raw.fit(X_train, y_train)

y_pred       = rf_gray_raw.predict(X_test)
y_pred_proba = rf_gray_raw.predict_proba(X_test)

print(f"\nAccuracy (grayscale non normalisé) : {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

predicted_labels = label_encoder.inverse_transform(y_pred)
confidences      = y_pred_proba.max(axis=1).tolist()

predictions_gray_raw_df = spark.createDataFrame(
    list(zip(test_ids, y_test_raw, predicted_labels.tolist(), confidences)),
    schema=StructType([
        StructField("image_id",        StringType(), False),
        StructField("true_label",      StringType(), False),
        StructField("predicted_label", StringType(), False),
        StructField("confidence",      FloatType(),  False),
    ]),
)

print("\nPrédictions (Spark DataFrame) :")
predictions_gray_raw_df.show(10, truncate=False)

## ML grayscale normalisé